In [2]:
from qiskit import *
from math import log2
from typing import Optional
from qiskit.circuit.library import QFT

In [3]:
def get_divergence(a, b):
    n = int(log2(a)) + 1
    divergence = []
    for i in range(n):
        if ((a >> i) & 1) != ((b >> i) & 1):
            divergence.append(i)
    return divergence


def get_a1(a, b):
    divergence = get_divergence(a, b)
    div = divergence[1:]
    for idx in div:
        a = a ^ (1 << idx)
    return a, divergence


def get_a2(a, b):
    a1, divergence = get_a1(a, b)
    a2 = a1 ^ (1 << divergence[0])
    return a2


def pivot_divergence(piv_idx, divergence, n):
    quantum_circuit = QuantumCircuit(n)
    for idx in divergence:
        if idx != piv_idx:
            quantum_circuit.cx(piv_idx, idx)
    return quantum_circuit


def pivot_fix(piv_idx, divergence, a1, n):
    divergence.remove(piv_idx)
    controls = ""
    for idx in divergence:
        if((a1 >> idx) & 1):
            controls += "0"
        else:
            controls += "1"
    quantum_circuit = QuantumCircuit(n)
    quantum_circuit.mcx(divergence, piv_idx, ctrl_state=controls)
    return quantum_circuit


def cnot_reduction(a, b, n):

    if a == b: return QuantumCircuit(n)

    a1, divergence = get_a1(a, b)
    piv_idx = divergence[0]

    quantum_circuit = QuantumCircuit(n)
    quantum_circuit.name = f"{a}⟷{b}"
    
    quantum_circuit.append(pivot_divergence(piv_idx, divergence, n), range(n))
    quantum_circuit.append(pivot_fix(piv_idx, divergence, a1, n), range(n))
    quantum_circuit.append(pivot_divergence(piv_idx, divergence, n), range(n))

    return quantum_circuit

In [4]:
Point = Optional[tuple[int, int]]

def mod_inverse(n: int, p: int) -> int:
    """Modular inverse via Fermat's little theorem (p must be prime)."""
    return pow(n, p - 2, p)


def point_add(P: Point, Q: Point, a: int, p: int) -> Point:
    """
    Add two points P and Q on the elliptic curve y² = x³ + ax + b (mod p).
    Returns the resulting point, or None if the result is the point at infinity.
    """
    if P is None:
        return Q
    if Q is None:
        return P

    x1, y1 = P
    x2, y2 = Q

    # P + (-P) = infinity
    if x1 == x2 and (y1 + y2) % p == 0:
        return None

    if P == Q:
        # Point doubling
        if y1 == 0:
            return None
        lam = (3 * x1 * x1 + a) * mod_inverse(2 * y1, p) % p
    else:
        # Point addition
        lam = (y2 - y1) * mod_inverse(x2 - x1, p) % p

    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)


def scalar_mult(k: int, P: Point, a: int, p: int) -> Point:
    """Compute k·P using the double-and-add algorithm."""
    result: Point = None   # identity element
    addend = P
    while k:
        if k & 1:
            result = point_add(result, addend, a, p)
        addend = point_add(addend, addend, a, p)
        k >>= 1
    return result


def curve_points(a: int, b: int, p: int) -> list[Point]:
    """Enumerate all affine points on the curve, excluding the point at infinity."""
    points: list[Point] = []
    for x in range(p):
        rhs = (x * x * x + a * x + b) % p
        for y in range(p):
            if (y * y) % p == rhs:
                points.append((x, y))
    return points

In [5]:
def encode_points(points: dict[Point, Point], bits: int) -> dict[int, int]:
    """Encode points to binary representation of x|y, that meaning x coordinate concatenated with y coordinate.
       Bits is the number of bits used to represent each coordinate. In usual cases, bits = int(log2(p)) + 1, where p is the prime of the field.
    """
    encoded_points = {}
    for P, Q in points.items():
        if P is None or Q is None:
            continue
        x1, y1 = P
        x2, y2 = Q
        encoded_P = (x1 << bits) | y1  # Assuming p < 2^bits
        encoded_Q = (x2 << bits) | y2
        encoded_points[encoded_P] = encoded_Q
    return encoded_points


def get_transpositions(G: Point, params: tuple[int,int,int]):
    a, b, p = params
    curve_points = curve_points(a,b,p)

    transpositions = {}

    for P in curve_points:
        if transpositions.get(P) is not None:
            continue
        Q = point_add(P, G, a, p)
        transpositions[P] = Q
        transpositions[Q] = P

    transpositions = encode_points(transpositions, int(log2(p)) + 1)

    return transpositions

In [ ]:
def build_transposition_step(G: Point, params: tuple[int,int,int]):
    transpositions = get_transpositions(G, params)

    n = 2 * (int(log2(params[2])) + 1)
    circuits = []
    for P, Q in transpositions.items():
        circuit = cnot_reduction(P, Q, n)
        circuits.append(circuit)
    return circuits

In [7]:
def ec_by_transposition(P: Point, Q: Point, params: tuple[int,int,int]):
    a, b, p = params
    n = int(log2(p)) + 1
    card = len(curve_points(a, b, p))
    reg_a = QuantumRegister(card, 'a')
    reg_b = QuantumRegister(card, 'b')
    reg_x = QuantumRegister(n, 'x')
    reg_y = QuantumRegister(n, 'y')
    reg_c = ClassicalRegister(card, 'c')
    reg_d = ClassicalRegister(card, 'd')
    quantum_circuit = QuantumCircuit(reg_a, reg_b, reg_x, reg_y, reg_c, reg_d)

    quantum_circuit.h(reg_a)
    quantum_circuit.h(reg_b)

    for i in range(card):
        currG = scalar_mult(2**i, P, a, p)
        circs = build_transposition_step(currG, params)
        for circ in circs:
            quantum_circuit.append(circ.control(1), reg_a[i:i+1] + reg_x[:] + reg_y[:])
    
    for i in range(card):
        currG = scalar_mult(2**i, Q, a, p)
        circs = build_transposition_step(currG, params)
        for circ in circs:
            quantum_circuit.append(circ.control(1), reg_b[i:i+1] + reg_x[:] + reg_y[:])

    quantum_circuit.append(QFT(card).inverse(), reg_a[:])
    quantum_circuit.append(QFT(card).inverse(), reg_b[:])

    quantum_circuit.measure(reg_a, reg_c)
    quantum_circuit.measure(reg_b, reg_d)

    return quantum_circuit